# A-share 5-minute Stage 4 — clean reproducibility run

Restores immutable checkpoint Dataset Version 5, runs bounded GRU IG, rebuilds baselines, executes cost analysis, and uploads durable outputs.


In [ ]:
from pathlib import Path
import base64, gzip, json, os, shutil, subprocess, sys, time, zipfile
import kagglehub, pandas as pd, torch

print("STAGE4_CLEAN_BOOT", flush=True)
sources = list(Path("/kaggle/input").rglob("run_neural_models.py"))
assert sources, "Attach the Stage-3 100-stock pilot Dataset"
source = max((p.parent for p in sources), key=lambda p: int((p/"data/dataset/flat").exists()) + int((p/"data/dataset/seq").exists()))
project = Path("/kaggle/working/stage4_clean_project")
project.mkdir(parents=True, exist_ok=True)
for path in source.glob("*.py"):
    shutil.copy2(path, project/path.name)
payloads = {"config.py":"H4sIABNoeGoC/51YW2/bOhJ+168w9BR3HR/JtzgF9oGWaJsnsqRKdNLsQUGotuzorGP5WEqL7sH+950hJVm+pcUGASKS38yQnG8ujK7rVrzN99GmsUi3q2T9to/yJN02Vum+kb/EDXKbvUT7uNG/fU22b3ncWEZ51Nglu3iTbOO2ruvaap++NnZR/rJJvjaS1126zxs+DDVN8wPvd2pxEXgeb/xTzt4IsUo2sRDN9j7O0s23+KbZ3oGJba5Z3BM2Yc6zCMiTsFmAMnUVvzV0tK/jxyJPxdcozfJ08W9drSSbH2Iffde1kHvWg3BYyIVP+PSCGikmNkmWtxfZN12zCSfvWdRwR/0ZcwtQhQcAmBR9uB6wSycz6vI68EgODcfrVzhrhiblSemYBtS16AXFxYniVQzXs4jF2zZa/vmW5fFS1z5ZIOp7Ab8g99cChNAPsG1qea7FHEY481zB3EcacjZRIyV5rAnPEwMXFskmkVQQyfZbnOXJWo508Cm1WPjLynagLclQz2uUx/sk2iT5D9zXhM3o8REubGSdvMbCNAy4Wos4zJ38mtFsAWa2a7GKgCV73LNn0TCk9jUX7vbpIs6yeFk4ckwJnwc0vIZfxVH+Bvwt4LgUUn7RhXmUxTlS45M4h5UjSYy/wK5D+Puw1SYCbSOYguug1znwFcxiiFZMCDmZ0O51gSyP1nH3GN77Cbx3CicBZ2NivSMQ7fME3QIB4DseFxaxppe4395t0lwsosVLLP03+hlZwIVfFXQCjgvLwD+NP4CtwXFZ+88M2TyGIARaON7kmsAKgjBetjfpGvzsPbmOR97FL9Pv200aFRKawz7Nmc34s4ArCqRzKUjpHaPTuzVM+NWPMJi4LG/uYr4E4tfXLOKKJy94oEEIiz1NA6tzTsWYUcfGKWRb3MohZlqLdBm30l28bb0k65fWJv3eWmzSLG59SzdvsB69pm/b/DwJnej6NTUtlZaAmusW1JIlpAsID/BwucGAfpqD+mfU29e1uUvs3+chh4sHwk9wtlu73NOLMu/Li7LpmMwdXruFjiZ1Q6IPHUp9EWK+kwcw2ias8eBZEM7pzOc42YXIsR688VhgBNXQnbah+SQAVUBJb+YjhSDB4AaybbTbQcqyvEcaAMvFhPjimRJkIW7usKBoKn0odRoHNgKi0GdqGv3sQ4GB049IIDjkQUTnb7tNfKM14OePlf73S/q2/2h0lv/9+LeqvGoAqVDWZlxuNYqanGylWOOPm/tyrilR1XpjH23X8U2332oMjFaj3/yiJP4BMqbxvpBxScb8qUzXPJXpvi9zcW+9/2NvoMhofvmiNY8uGjICDYBVSMLeUAthhREHJwRGb+XEACuL5E8f3McgJrjnwBJEh5w1TI3MMEAFGYWeg/w+AzzCPFS3awBtyiZTyCFPwvOpK2aEW9NzJRUITuEQVxW8w+7uVP60fYak6htqOGIukgmGNiMT14NSaUFQT7CY4bymjRxMMTwgNj1PSUYZaXWUj9fisBnjUjMkJfqIXc71yOtUiPP4k4rVkav7CadE7Q8yHlxPHRNQKIns8cIlj9ln8Kxy0TUUwKzAk/Hn2l4gCsdUqJplsIuVC0DWlLiTY1VmfGsamuV4mDTIzHfgD/sXVUEeABts7E2kMvFpTlw+n1WnqZZn5LOgQQAWkIkjmUCuyP6Gia1cgk3Y3kw8EecBPGb/ivSN2fnwARjcrLVrRct15vPurTG8Ne71g0GHPdCr8P6t0b3tGFULh8GjyhaEUpH2iiWwLQ/tkKBIm3KfKjn3KwUENKCWS7C7CuYDl69q61Ywtd8Q0jOQ54nBzT0B4qY46B2Su1Uc20S2681SEhhyWQ5O3DnIwaiv5BTjRejNA2CS5dmSSDd69p921zC6xhAlspf2wDDNbjUYDjv9wZm0Kh21ax7cGoNb804vc0nRkUJC8B6whtVuu2yNC/ped/AJEA986ljzXi/DQKaay8kJ46qyBrFvPb6DOEpfkDJYAGF7cN093L9PWFC2c0VbddbfqY4DGk5s7mMRvS2T4u2Gx0JCcHpVWMnAEyZWIj7DZKkEz3uusv2XrIT7F1N4SXqQyYo771cAXLzgEYeMqINx+3AID5n+yqWuMZM1CeYGxdTgMGV28Lkwlw1Zpb1Yw9KlElC9ebjRTeOj0cf2AGgG393q2yznVYU8+dHNbg3bq+noFfPNw17KU1tISVlnVL8CD7UcXkIoJssyOGmDg12awesRXul7HIEDBfbExSdgRMdYFvuSKtJv8X4LfWaOmCQTq2Sf5aU4trUiXYll9ENk8Og6mVqkGahqqheUNQ8CrD/nG752Fg1KHXgK2zrVVj5CqNhi5rl8qgoXx1JXjQeQLR1Hsb826c8xNyGtXJsEJSNMbIRF1YwoU8qLqlgbAquqDZnaoTUm1p55qgSElNqSYZ2BMTTuyjcdTktvlAutEjKsvu7LL2z3ii+zWXAvnM7HY0c2LJK10DNMVGKQlXbm+BACtg29SlH1zM6wPuuQ57Idd+k8wLYq8HxvzlVyLicduF1XVTReVNVq7YlCuwPxSi3ZocFSR1oYyfxTmO0Z9wNtEsxPdjPoyclqF6YcHkma8NwqTck67HvWtOiUimnYnCNfYD6mEUZVLhuWy5Y/F3waUGKr1xeaYK4VUPkfn7NbM/qlIHoHk/KIOfiMQ+tQxStU+XLmPGCjucyShZvr3v3D+KIdQRR7ZC9QVasekOgIBOyU5O0cTdPPHHddL8bHclO4Cj6CABHQxgXP6pbqAGVRZh9wcv0ZZTQ+QEeO0eSGkP4RbGHhHsmNYI+v+NcsXp4yPbKJSxwVQMV7HC9XGVG8bjWGwFVkyGQ0w/CGVo9xGpDqnCpgVBmtenlVCutvtRNAcRCJwzKoGoQrKIjpEnNXYpA0DDMDPM5thtX0oKzeT19FF0pLrNk//LfDQmJhC44hU6SSYfEmufJm+B93Y1VK0BUAAA==","run_baseline.py":"H4sIABNoeGoC/61bbXPjNpL+7l+BY1US0itz7GSnKqUsU+UZayautT0u25O7LZWKRZGQxJgvCkHa1vp8v/26GwAJUJTsyd18GEkk0Gj069MN2HGcm6Zg66Za8oRVZZalxZJdpMtV/fnDJYuKhN2kyZKzeSQ4vOOClQVbZFHN0qKuoiTaMBHl64wL33Gcg0VV5iwMF03dVDwMWZqvy6oGOkVZR3VaFuLgQD+rluuoElz//kOUhZy/jupVls715Gv4KV/UmzWyp56fFpsR+xhlWTTPeEs2j+p1VtYw/6D76jeCu87pcul42+P89Qa/sUiwdVbr90WTrzf4rFjrR2sQBzzAccmB5Cgui0XacuQeMPj34fR2cnF+NQlvTq/OvlyGt5PJ2ch4Orn+cnMXnp3fjNjZ5OP5xST8+OXr1d2Ifbo4vQs/fr25mVzdhZ8mp3dfb/DdxdfLq9sRkVYPb8P3l+dXkgRNOju9A/KK6MXph8lF+NPxZfjh9OZ2xHDo5L+uJx/vJmfh3c0p/Lw9vby+mCii119vPsM6pxeTq7PTG6D1L5gkRyCF8O78Ug+9+XJxEd7eTa7Dyy9Xd7/BuLvJ7V33g4j/a0LL/n56cX6mXx14Ul7zJs2S8H2eFmES1WBTtZYd7eP08+ebyefTu3bbBwcHl1/OYDt65yxg08O9gmKHw6RmQCvhC5bwLNrwJORgv5uw4mCphXAXPCKTTdJqTBbnsaNfQc/+GbD5qYpyPiYJgJF/wD2ASTP+xOMGjfooi5ZM8EKkdfqQ1hsGJskz1gi0VlqHgcPUK84K/lSDL1UszkrByWWQalU+ijHLUlFPzSVnuNsZjViUFfkFuB0zePWXWTl3nUMfXOnPhteOJ9mkKUgCKADFikdJqIa4SGYElps1eSGCqROXCXdGzAGF0Ged5vRJLDozb4sgffoCtBY+RFnDhSvndCOXVdmsIaDosfR7vnHlGt5U024n4Pzw1Vm0yszmZ+osHhNUI1h8rpQK6jhxUHjF2s/KpdvOMHjzxSpd1O6R7S0ee9cfcOJ5/RVJvbBWEkqOYCVzAztI96l0vO4nc9LNBLHF90ILSAvET2pf1NUCf7nOd7+Nv7scf3drqAPNy4/Wa+DYlaIFOlNJzE/BSt2+u3sjS2bb/zqz0eayQw2vEWJ9aQKpLckoI5TOigYNcTeOahc3NmLpsijBHdIi4U/BXdVwT7l6VoLZY7LSwcYFl+FxXVYb6eQjtuX37L/ZVVmgmePH7ihwAbQZJB+25tWRqEGUMi2Cl9UpZTrKnrwAz405KVbUkCkxialkiZTQGVGh6E08Id+0fL1leNvTWbqgYX6BbvkfAXkw7jLMoyJdwHLdWFoLxkMalkt2YaKKUsHZpzTjV2X9qWyKZFJVZeUunKJst4RU2ALGCGTqueXqRZE2Yo1SzXQo7HjW3oiR2aD+DDch559J2nWJmuRk6NZrD30AqORRlv6buxaB1rmGCcjXrYQMg2CpIIGhIYx3RMGcA3JyX80p3ghwU9Bzmtl+31iVj4GT8UUNEyDOpsh44AAvIewCPpTo6VUXEqQ3VVFxD4EBuC8i12PfD8TK7rVB5vuA/Y8auq7SmIMpiXt0oSVI8Pv2HRpa+GcDU+qNOcT0UjmUyM6sZDHtMgxKA3RXcbRa0r+bVOXacmIFSsNHeA/u7qIYIFmCMm95lXJBLkqpM0njegqBcIQv77TDzWatz34Eg6zBFx/Low2Pqnf1quL8KC+LevVOpE/yG1MLwSf8ADkfIchdRuvOaaMiXpWwTB49pXmTg/CJKR9gjeuZZjjSL6In+0U/6+9ivQMAjyvwPoaC6UwRADgAKYicMEzyxP6mohX/slhgwMN9isAAZiAuvQZYbR2BPDciMHIMaSyE9QGUBcYSfxuYNgAdYdwWMcmhSbjPJkleBCZmfJVREGzd8tkt9Jf4JFqSTYPsDi4NzPsqkxBRWtq/aoMZW34/B6u8H8zVz44UP3HjjFu7c1qtwMP2+45o4hhyh+HGr5F+Jwm133cR6iSDy7Y/RuqNYkd9felkoE0zGBZnv6iwcj2KQwUCsYZwE1JYkbF7bGXmkXLcMdvhTBQo6gZq1ak9z/f9LkbcZhD20O0VOcrikRAc7CFeVWVRAqJM4ygDwA+pjYraNjCQLtpYbGUof87rR84LV5KdWqqdaeanhmpn3mwgwO8nauraINqpWRNFNb2V0U7tJp9a5ZoiWjry7vN8DSUQKJxW7X7iDPmrDz5+x8QgUYejexBK+oAmaiAqqN4iSmQGNqJxaeF2AxRt2T/k+t1TDM7wFNmwH3p9fk5J3aDYHk+0dGcFG/aQlgCQeOJYNkvcqJw9ovWUBassDoTDXHA3imtIoWOsUookqqpoAxZbcTRfnpiPyXIXgGbrLpVFWdxklM2grOwIs3L+B0Cz9IFD+Qm1FOKtuMQ+C5h+AZvoak6VppGsC2vlPCpcxRNGNc2Ixw4P2Y+eTscL8MEM+0PLeS7l3ndDYsZ+aLgedW06D4WfZoImTrMN8opOiN9WuEXI0yxqaggjkD6TrkGFHpjzmlfC8MGNCdbKnGl2davhAmbe8CWgDoAlNJQ/xXxdswl9oBQjgc/6dnHTFAhdlFW0TFTyMWuK6CFKqR31C+BbcBkoD9rFMY5k6bzM1wDdiS9YgRaYoyvlgIXAFkAedgXSjhj3pDYwDJWd8ehBovTpyfsR++lkZkgD3ldoM/j22D/+acTg/59ndj7CUeAXhGyIzPHxiL0/Pu4Nk9ULiB/Ll2enaPJQrg1JQH6BxABfKrS6EJeFF/ixD/U62BmKV9gkUm1FmKSYAXIQGAWIOoLSA1/8+P74ZYsYCRJYsrTstm4ROJV8CGoGDApAOSlzDG8Argc7d6/Vr+pfEf5RzkXwd7B/Xs1LAcg4ODoZgfNIIXnDjPrgTyp82Y0uCLUqqpmIfpuKiKF20gCoiy0q9lnVAJQN1NV0vbduSvGoYoEi2uPT2+YJwjJaLNZPZKJgUJLNf9BzlY9aZp0By7L9YiTpBUz9etZS1RCmozWWS72YMW6LEDD0/GIEtAp73P/v0ewTJIsI6D9ha50vIVzL6IV9dGwBQiiAJwLCDZUatDz4ypGMe6A0Aa6Y74ts4p4czIfA1qBby/h2m6LnnNOzaniCZEHKRE+jRv/w8HW6pu5/25JXv3eMrvi6KmP0sa5bf1tj97xKbgE68f9D1BVxep/WR7TSUOTtB1cdDjTL7tR1pLgqcH9LVC7oEMLAchM4AOfTqHC8nZ6CjMBGkIS1MUATI3hJBgUvSaZulK1XUXACsdbzlAv/def/aw7/DZ6snEa7mkPcg18h/3v8TVXrsHKYxi71MAfcpodiSmm4t2tQZw4Q7wamn38kfyibGljJNmhDYBu3cXq9+UGwFc/WvGp9AkJNBliVlkN099OxYT5yI4ijomIb89Ac7HOQSDAdzHxkHyp12FGl39stEzmgRUMcexpgM2FHRQwVJ0bI6KIFsTEb9QKOdQAwhPfaTqIU1oidf8RjH8QXCY+xS5CVEEXECt1O9aJaed3zDebrqd3tVwg+3mq943APy9Fs4yrVjgAUxFmTqD61CD5FmeDYxME+JJi+HOZ4PrZysL9k9XeUjcEgo9Mfl2iksiHKq5iDV2OrMVDjbHaGVAYYcFUmgbNIK4H9snVcG51ESQX2TOLRJwMKViD2jXmauebKh9bRnGef1IFABHZx3bSo5QKS8ADDeklQj1ANTimuvl0R+lYOvsa+6a7dK2VkUT5PInlWMbbPXai5r4xX79jnf7q9He1mAP8d7ad1sovADuuQMQ9GkAGJOgnJ2NJ42tqLQcF6CoPdJCkXuqsCqq7SWIJOPWrc0XbSOK26B+ydXg0ihfrGgSMVFbbCu9pliC4Ukgupdi4GOqmYlk9YKgTMx6sHebRNwLdw09iz0IdiGPc10paiWxpRgR1UbJ6Se6XxWEYFihcQJDosQUeMjVjD9spGtOfx6C+IEqIi5lRzAIesXLCYZ3yO3QnMwLUZLuU6ECq3I6VzhSem57/jIfXv55P/7M/5lcqF7Wm/nX/+zZqiX3z5p6PzQlNgz1oiDregACmDYFpD6h23B/lT3/dHe9DVYNP8rchNVuO9ZwP0ZPMhxOng421Vjjlvx0HRAJHBmC/rkDcGfQKQeIiksgy1pRRmK+iH3S1p1axyd1ueSSG7ZmPCABOhOgumvLDrNNhydm0Xuw79kD2S9IFVpuiFdH9z13SDN92RnVoEZmbGkOPsrKBLhRbwEI0+3tHbsNU7fOyjb6mUc7xa0hbA1kw/v4f/XRA7bEZQAhoBEE2h+ijvjXzUcY0QrT0gs7l4xxaOssPn1h5fwudCnt8+ItZ8MXFHe9q3r7ozc1A7TG7KTwArutp0/gIzvqTjWHHaCnuglkHEJIVhzbPOx9tsIDO3hnNDQyn8v2Hczjjfm/uKlUvOOzNRJ4F4oisVjCDI1ZctLC9qIZOlLj30edeqY9sXXzqc1WcibFWgH0hVyO+vqEQN64i+ppfeokYK30ui1dfWfPlm7+S9SuwTPHhbnp9Z7bJQ225YlY90IG+Lz08hsOIhhevZTQy96kKv9AyZxO1oTX+Q33+YeS+K3fbVQLCV+Nx0pD061aeLZstQJR6CTdKDwZJaV4YAT44PzworJ+pjirYZhwiHnnnt8ZH9UmYXfSjUmwiPvC3iULECE++Pj+8t+gBRXru6RkS6HjBMx1uEFMiEqztEvBB4Ch+JOE1V5MO2nHaOjpBEOaEByhD0GRBN1o3bBu4pwIfAVcl2JmGsFCpK3RA1StgsG8xJ7aP9M7vTsSFYidcZwX4wYQkXMaUNNtTg/mMouKEYN26/dQnQODtfAejgkGYAaSZQeVI5eiToJN4sQfMSUmhZgA/XG4AfS9BAV4TKld6aLddZ7UsKLnyI9N88cE9ORuy96lckUZptqJhoK6WuvFViHKi/+nVH20FXGZBokccj/a6ikxQNb0f+UOK6YSCPlhQBcx0F1twfj0fY0w4BuqdlIoL3XltSUPsmoCW6zUdPK+pcHdOtvbIKnHkWxfewPXz8mCb1Kjj2f/Z+odEZXyK2Ur82RNF1fjymawy4juobON0CNZ5RQETdgGL0RBE9cBC3K3UFAEC7B/UfSN/+ulgCE8k6DU7eH6t5dLPP3a25nzvFITOijXzC0J6U8Yi1Nee31M9DOqSlvkWH7bpDXSC8+VHdc1BE6ezQ2ZOS+3WbUtU2nZ5iLlEjwPJjVCUMqLOsXCr3dnoa/UZ9qYRieuJejXWFWnvZe/ic3QwUQ1gTWTGxxJjNyxJ7sFJ1O6/U4U10/sCrDbXGMwjpvfqGAsxjBSmXRU2S1lgvKm7E0F3arewn79P8wtCMcfIrl26VBF8Z+otk5pVFpWlSHSZ7F/r4EHKMeaYKRk9NY/WYGskvrV0buUCJBIybF03OsXPt9u9K2XfksPESVXVw4vWuEFln1Qhd+zct9GrGnQ7acndRZQhSHB7KH+2FlTx60hdW7MP6nRWLQhgQLfUFFfvovsUgknRvxH7ShE8k5e07Acb1FZQ7gSKlPrqCLRWJ6C/fgn692ilgnU/i5dBFFNfCoX5Uz1FkZ2qrndDL8tgY3OqmaOZGAxodGUp568HfwEa8PlOd+jtNHx7CixfPcDE9KI3hqeVR+o186PVkiHBZF52BUXTK2B5Q6fmiOlPBMyFqlWp/mI39vy9eHOywN2JlQImud2i6MV0jJpatG6ztDnbeUO3AeDfL2uHOmd+GgBTb2CyIxYMR4pVU9Ht46YzMEh/Bb1wmEBICp6kXRz/rezPxNq3t9P5mcjqBb9HUNY06hHgzQUs7MtR429RlmKLXbyctmhySN+JFLbUeKtiCGn60NP6EQIXWwNUhD0IQIhsHz9iw56yMMOhOQhCGcjxNHBnFbFSE24U0TBqort8wH2tlezJVz2+Y2RXA9vyd9fEumoqF3rrynbXIt1DGwlCWkGEn+K2qEVXQ5I5naXjbXhZNlinQHepBbzdJu6ZqqygNh7yBlr+CVHlEV8+sior+9A1zg/4zOP+0WkIaL+preqPRHv3woyQJI/XedY6O8Fr+UUKCxoOoQP4tA6wVNVkd9P8kbD8peUl9D7X+35ztJSeFsZvawJ/B7aWHKfLISJHonvQtcESNkbWGKKlUBPMoFktK9IG0hNsP/jbG3f4rEZzkyycpZFT507rMT4/kXmnM3rRKg/u5Xu2a0pwR3wE4Yoloxh9E5gAWwhAzXhiyIGBOGKJRhaEjrUla2MH/Au3wZCXOOQAA","run_stage4_attribution.py":"H4sIABNoeGoC/6Va63PjthH/7r8Cw09kQvHOV1+b6MrMOHeK4+n5UVnXtKPRYCAKklhTJAOCZyse/+/dxYMEqYd9qWdsSyCwWOzztwt6nnfBcy6Y5OROshUfnJGL8ReS5pKvcHRB4N8i5bmsCMsX5OrzLSm52NSSybTISbopCyFZnvDI87yTpSg2hNJlLWvBKTWPYWVe6AXVyYkdE6uSiYrb7/+tilyvL5lcZ+ncLr6Fr3ZSVc9LUSS8qpqRbfNRphvekN8wWWaFBDon7ceorrjvna9WXrA7Lyq3+ImwipSZtM/zelNucSwv7VAJgoABnLdo9i5Esj7R/CdFvkxXln3/fDIZX/78ZXJ5c01H/56MR1cjevfr+XgUEvfRr6Pz8eTn0fmEjv41Gv8nPCF7ftwFd+dXt59Hd/R2NKa/XV5/uvmtS/BuMrq9e5mMXkonl1ejmy8Tejf6eHP96S4kv3wGVj6dT87vRvD/cgwjwNyXMez4/uryWg3tJX43+md3HQx8GV1/HFFDgH68+fzl6voAb3eT84vRGe2cZDT6FNoH49HtzVhRDrS4RZ3TOat4lubcCv3q5tPos90PDpMVbEGXGZN0wSTMlSERRQYrVvQhzRfFQxWSqsxSSZeCgRU1hHNeC5bRTbHgWdWoFFxkDP4BZliIEH0iJIo4E4JtkRT/vebgE3Zg70HdM0s0KbFI/+CLT5rB4OSkKwIjQyM7EhP/u4OCJV5a0UQUVQXn3YKxn5ws+FJLIVnz5L4swMN9dLSh8q+QLPjXNOFDbciR/haQwU9E1mXGp3o4z6OrYlFnHOaniZwN1cHA8ce8ghmcsCaKtNuQh1Sui1oSwZeplCBzFSpwZcm2yBMcRtPHL4qrEPyypFmRqKARa3ZC8sDT1VpWtMizbfwLyyoeKDpKPUDF1YtviE+9NC9rWXmzgKRL0oyqNd6MxDHxYJ1HQMEcdblnYbuJYpFWEMw4RRG0k9sxWPDBzJaFb0QZ8a8s8zUlwSE85npKaDkyOuKPUvANp2CVsKzyS8GRJobOIcSgCO0EbCqEKFPncoiRGu2NL9RHpbF2VqOfO7YBLRIwS5ZlW6LMW645Qeo8yRj8AxWUKsTPCykxCjcbg20kacarRm0iX4GwYRsB84sNWMuS1ZmkMO4jK3D8NcuWMEcxSd68Ie/Uwqx4CMkadKiXAze5BMruIUMyPRwwX/Kj9ueUDMhBOkahkqXg0zGZAivovnmR/8FF4XJD/h4j00FIjsz5KVZnCmaKarIuKp4jWRBHBN9AjT5okz+CUae5j5KBgMRzPRgAbcHLjCXcWDRZFoKoZ/BXMzlzzQY4gQwDnsFz+PX1ftbFKyeS+GBxNe+ZjbG2oXJg7eDK9SY8B6dpDOa8LMFOWidG+xAszcF7Bzpi4lYyrWSaVAT9sbGOKmEZR582+4MmWqfjLEdHfNMOqele54SaIzRSqtKvjwdgOZUFfvf1BkHEKrktua80UzD5l3eBlUILX2gDX3zlbjbAOZHMSsmVwxFL2xcowQMlLyvjjWvOhJxzyAf8KxdbPXqYoBYnTY978LjOyUdWynpDLi/QMObgWguQc7Ku8/sK2EHcISGrkYvbL2TDN4VodSKFoYQ/GqQoWhGTUti0dtlI7cIKTa3hjwkv4bGaNRICzBOwD4y2JME0IHgCj4jB1BTf0zuQtAK1/l6nAkNUmZbAO1hOlhkOvEDzA+QUNQjRaaWmQP70TYh043qggpSWe4QGoCJ4Ui+Y1/Kjo+9K1JGyWr8JyKCbhRuKNxxyExrrnsPr3WGpFvGQZGDu01Y3M3Ty2Qd0BCGVxePpo00BYLfI08Rsge6spqDWIGKuuP9WBwBtecGOxQTtQeZMJhgt9dSpojPU1L7vL5s56aYhgArGIKcPqvSdzmuIG4pySCxyqmJt0BjdKpql92ZG8Iqgm1Nl/rH6GyrvEzmgJkWAVhCL4h6r5DvyruVRyzdiZcnzha84BseSLFn7QZSUNfzVcSBo1ywKQHuxiqgHxNGRcbuwFAh9lt7lRQu5jQ/GT40vPoM3YcoEvcZPuNfzmyeH3DPxemJZejxjZQURp+LgiIsqfupbAwRCYyrD6HT57CForKt1PBE1D45GeBRPE+HrzYYJkCmEtloLqxvh9RlMLALcBRhGgzwVWspFhAjzF4S5TXD5TaRQAJ79MJhvB0sQYI1QDrwMNlqloEnC5lWRgdUQaz8ICwwnDjDA8E5hrk7v8EHzF0T4wGePaRW/1ScFXUmY5XLj29WIbbJ6k1fxMQTc0okgXHAh0au8rCju5yy5VxbpqbTNtM8NUH04PQBTOQ3JaYAwpbdcy86zQtR7zLfUCiVWK6YYCPyjzM0ikA7YLQQsSbXR+KxKwL4hg7rQFZi7TxWeareJBCgeBOJZsVNH7F5gHlOUp++ZNWoY6gaqkIPfIQ5Jdg0QD8FuO7SX9gwy8wsz9MFc8i8KUEkZkndSffW1QcI2SzOXGqd7Rnt2t6K4DMqACJZ5oUZEeyT3SsJGTNSuOkRUeyBubfwNewLK1Qw74G2u1e5xsWsIF05lBFsIBbYxDUKdhFCKYy5lWeNshrZyOsHBTOTAcFKKYgkQuXUxBgJ6NIaI7ECWK+pyvvV7th+8ykqVYwbRRJdjmYyW6QoY8uGfitv+6V9C8h5dBR+mm2pdPPiahxBQQMkTGXuslgUIMwF2Ym/DVhvmBQ3BLSDE+8rXTog+qFcj7tWfIo2D9Q6P7mwwqLMf4Bchsvbhs7+F6LtnQUsfYkUh5kz4GZvzLPau4ERtvLq88Cxl9dz3PhspQeYTxH8bJ7UAf5KBw7LEKpNmbAuq9c3yin2F8nXVmpmHVpWuGist8xXIYFGm8en7t2ZRkgE2b4u+V2nNg4n0/cazujmomR9BDlGjGjRTH5ZGptTAjzruoJzFPRexh0rSnxWFdy3tl4Wjd9maeX0hD4nh+s8IEZa+31DrE24IOCpU455tM5K2zcgXwf7Bkugo3Gmrvi6Fpv4+0kfZm3hBjhX6P7ZVbSxoT6FajE4ZnlaiLtVHtgSI5Z79W2pzhHw6995xTN9OMRvgsnsICLrUQsilqRYPqlKeNYhWp+iw5TonHFaoVrLfbb85cLZa18ulrg61JsB9cZMPzZPp0KZ/la0gUjun9PfMapGdVKUbrOoUwXZJo2fYTNufcyJdh8m1NZhC1Y2+w/lxJK+3hjh19vbHv/ZW4Y/Z0EJcZZ1mUQ/TIwEXyB8EwEkBrpnpLryrTNQuLIKnfjvYQ5SGncCqu6WKirZcPjXYYmjVDBkeV9DWFOEZ9locbp47mbSD8JB6FxPtkMOs0oNJ2tGxG6sTut+tmHXvtZ+VQZrpkiUmL+94tZu0w0M1vWqaUafR1qfyYtUPf5KsXnC6ycohlOxFdjgWnNvSTIGFZfrIEQuUiOpyNP+vXMUZZYVY/EjAEgZmNd5vWjNxv7vto4im3gKU782CaSNA0NwpVLAlsI7e4LTA1YoujNPNbJzWbW93pob9/r9eC5LawMo+NsNKS8EzPBl9OtD7f/baBmzoNI37/WyrcwUCn3DL56iUntWvyU1ChR9QgeCwHI4Oh9m7ljpNvsjMwwTdDnuzfqCEYhogkdphXye3a1Qv3wvslweEiUZ8RjWmyj/c+Aq7UVFrbuqhEXmz6dvZ1HI+awPlK0r/Tv8r3C3Be3yasm9PDetMtZA6sA0hqKA7jtRGKwetKwvJSnrM0tTz11haVoaK2GttzW68x94UmdfYXEPjz9idPtlB27NMfKP9fZMNOrglPgjLULDOVZmvIk/g2qF7ktnrO/6OtsJG5j1irom5KnIusQ+XsnvO+WLp2Ridi9ReXXmqjFcgXAdLWUGesVfm0TUE6KpkBll2680agNgjGFW2VVnCZAQALPjNXJ7DZ/XVdFFt9sBdIn0YSMki2tzDX1QRhhHVpgqBNlSVtLjXXasPhkRzg6e/Kn4jV8Soc+tF7j2snqlHUhFqFmytDiNGNm3uVxO0VKjNODr/qCfWpRxq7YGs1nesSs3reIYZ0lFNfd4JbWrUCUsWqBjmkgJiHLYzD6iu06hTelTt5UqK9lr15zrNFgRKrRQqLfU2RgVJPeP2CkZvRcxWjR7Nd4S31baK+CNPwPjmWP94g9rDaC18RD4+pdhcoFQ1j4rsK9RVTtT3BgO9w6DxBVxpMAESG1h5D0C85vGOHroUUVn92Vb/mqZR/84kxyo6FLWC+9NbtWuqWvWe0VvfEGCC1v9A6d8l5NqFJqXMojMFB7pMNeYyUObizu53+zt3X1Z33xN/CmSMfQ3AvrwZ5sK+1elr6+nMgcnfGC0a22rfhLhPyxKbkkuWZiBy/eoPzwF5JjjMyBpwhLrtACn/X+Gj2aFTXLZBS7u7ZswpIw/euHxw+/vm1RbIYON+i39f933nokxt1bxxFKFo9zh3A5JDfV+qT3oscRnBxc7hqBmztwfBSbfv0D/Txxt8+Wcy6h+L9G8hvD2MANj49rsJ5ybQd4Qy0XyPHstU44v20UeWQd19q7+pS8Fg5+JQsWNsoK07TbgZNjWFx3E5DKgbX6AQRJSqJjh9ASL0r2WQxsGzP78k9rt/XN7eAvjpSx1gXVXk8dMue/tEiW+4qDqqwE374SpaZcXct0jiu6Nt8aCDkRXd45eyePHq3uEY11I64AvTvgPrnqe5hay6ceBPLXpFxFOqXje6Kn5CR1V7A8RKV3khuL6FcI5sSVrI1DuzbQW65wRWbfl6CDM1vXlLfifl63n7tnO3aiwQX0L0gugB78OoBBTt40i0AKQNZa6ZpRnJZdy+cKACLcVdfBVkd2Pv0GgeBrFF1Uw4F6t6A8Ru1RPb7lVfIrZYIE313MdkrKWB11q4Vex9Dx/R5GLTelStvlj1/XV76jQkPwbBcaK9DG/pHV3US/tqjW6l2Lv+jrnvpeEAAWe9PUT/9cPjpDqIYR+1/quLR8l1AMUeajtvIvbsyTtKvQEi+t2gKp56SYmoTL/EAP/B2LxZu516fJRkD7rsmsTx90aP0zZg5wWi+LLpUTq7gOg4xd77sK9wjIHJoQMb6p0tTg43MHa33v9K7FEOXJwGQUhV6bGnXoqkEjzB6zZFNRE3akAggShus4Z6p4XSDUtzSs17LXY7zBydlU2F1NSKel5g8KL+1i2a8LUcTBmIBjWENMDRLD35Hz0c+O0aLgAA","run_stage4_inference.py":"H4sIABNoeGoC/61YbW/juBH+7l/BCtgr1drabXd7QH11ARfx3i2abNM4BxQwDIK2KIWNRGlJarO+IP3tnSEpWZLttEAb7CaWyBnOy/PMDB1F0V2jSMF3ophlWghi5LeZkbniBZEqE1qovSCZrkpi+FeRkrXluZi9f/uBlFUqCpNEUTRx64xljW20YIzIsq60JVypynIrK2Umk/adzmuujfAyNbcPhdy1Arfw2O38Z7WDpfZJNWV9INwQVbevaq5SeAH/6rR9Zyu9BxVO+b5Smcxb3fTjann/891qzf5w8+kzu/p0NyUfr5f37Gp5v1yv7v2b9fLm9nrF/rK8Y/efblbr6YSc+Vmv/j4Su1/+uHrP1qvV1Tr2p+tGsR03opBKtDbc/O1qdc1aQ6ZEVwUs5+xJqrR6MkdBJRrNC+ZD3Er/ePfznci1MKbSU3JzfTsltRap3NvJZJKKjBQVT1mXNZZpXgqaSi32EJbD3IU3JrM/Q7ySK275R9wwdx5CFq9BmvCiQAhAssu6EMTKUhjyJO1D1ViixZdGajCYcJJV+onr1EPHgQDVYD4NWRCw0IqU4iPudO8BT6QzJsmLakej3yQAhi+NsFF8LtAyc5KJAjvJrxYkSsFqIywruZKZMHYk7jyG48E/SP6eW7qBj1pAWMJGZ1I8sMnZvJ0SAH0F6IVUiG+Le92Ins4NniyirddtK4aPGBw6WI6T1CagpeSF/EXQgQLc/YoCv+wl9kW1f8QwDtZQt7E6c1LRm5/mb27mb9bB869wYgoSXjSRRio6xnJMviP/CiprLQEhJTePLCt4Hm17Wr5bdNsw3uxLA6/t4WS3FkB3FYx0otsEE8/gcyMMDYZPSbQHHGN0ALuQPBdhmuqqDmH26LWQT6aBBj4k8wFKp8RzZE4Q7xuIwxTX7xGfFrC6vYzrdQGukkoVUD5axgVtvzbuVILJMB2IB34N8pvshH0SQlEvDg6izWCAttG2NTG8FSoFl7fBOQPUcaQMsTnr4jmqqjqBKqc1PwyJ2io0CGHxje8t+GcfoIA3dV1IqNSeDJVOhe5c2z80yiFr41OIPMDkTEmuq6ZGXU4scY+7A/WpmzpCLz7ywoh43jEV6wIaiHXnWGfIW5JFzyj3kqj6lyjGCs2hLMuvYj5geV0Z6bqDZ8Unh4uws2NEkreQEZqeFAlnZ8cQ4JXrEzS1h1osopZk33/YKLON4mGRgepCjyb8ibyLE64ONJ6fnKK5NIL8VRxWWleaZlEpjUEYHVtkm4858a6PCpoPfMLrGnBB+2mnz5HzLpp7ZxL3dNp3Iocc2FVIY49R+ke03XRObOOX4KNLu0gHtdDbEA84Gs6O+8CHjAKk9480KNmEo3vxjTvSYr9loQ2hCdT1rLlfSJRKbqq0KQBg/sB5D9DAYH5A7HhSn2+1hKTiq8S4eo3+6RI1loD9AwRb7B/rSipLzJ47vsOoQHZVA86msx23+4dj6jpy4F4XMlTMFfPeUuotJ7PW3E1UCq4AmoD07pUTRrhyg9ijoCODFfv+9yEhja0b66n3g5+dIJw0uJMIOCT0C8eqEL6K5Rq41YMkMtYVHOSq5ioX9N2UFFCTvPnxlHx498fvRyAOp7f4c8dTfwbOHCGrXsPGqZ/7Q37rtIFbR1vjZF83NE46JAyh48EmFPyn4dgWLDhB9rBCS2H5uAYel8dQsQ+9svi5UscKfyu0AVL48tebX1OhrMyk0G+PWnFq7A0s0JGawkJW0JbNpi13vt7DX19YtuAV+vpD2I/9s1Xoe3r32NOKdOmPHTBiuNnCl9EQE5j2gMw5EKOdjJPPEAhT84DygafrPYwobkjLqkZ7GJGMlxIqPhRZ281vKZdAhP4U17mMpyU+MwyKdlI+wm8wEMhgjevIU2goEE9WPY7nIHD17JDpdMJo4DRCmMJAC9tHI+5oXpocCQ57+wz3KgPkul7ltTCZts3WdT9AotCIt3DI1DNk8bseC1yfX4yHjFZNPOCXDytoppGWae6AUMj8wea7MhoxK5PWuqLhryu+Fzrb2+GfgSkygwaNsXHNMXjx3Hnzwp7dmS+J1zJqHmPeoBPTcHIS3ruXm+EFYxuf1tRR9v+DPQywxDydzJlR/TiELE4mHG+kOw6WHC5Og8yUS0KosBhxGpVFDQE/586x//Sqa3ypbwx+aJTrBueY1uJ4nEesqwIyCUb0b3Onw4CxosTZ/HLYnFcQO1DHnvH3S3SiJTSODvVH3IS73wlqnvHgl6S20X/lMSElrxlcB9wFfOGZBHgXCGTDcB5uK9GJoMP/YnDhpF2nkwprOrY/GKCO/pIF3M8wyASurAKvp2dEzh/lnMcx2grmsHzsqt27c8LneXFhJGlR1k0d01B4LgDoHFF8Ci6QwtdzV8IZClNXvk+r+jzclOGlhiB3G5Y6hzqm7K1bCcOA35bwNEWdbp1Gs1lbW2YtSmZgIrIGx95b12v8ZV2kvRJ+QZkH3P9FlY/X/6IBm8ipPESWQ0ddjL+weV2V4Ph91Cvaxt8IvaoOSsdlVaNvhF5V5GEHevYPFXwwCxg76sbdk5uU49+yNnifbJW75cGcFTT3wQb4Az4yx0XGHBsZXNmlYizykMNRoy8RT/4NkwbrxP8TAAA=","run_stage4_analysis.py":"H4sIABNoeGoC/9VbWXPbyBF+16+Y4CEBHBCmvOs8yMtUaW3ZUZVXdklKtlIs1tSQGJKIQQCLQxLX8X9Pd8+BwUGJ3nUqFT2YwBw9PX183T0Ye553U4uNnHzPKiljJpo4qUO2Fkkq48l9ksX5PVvlWS0foFlkMatLkVViVSd5NlnlVQ2NIt1XSRV5nneyLvMd43zd1E0pOWfJrshLHJPltcA51cmJaSs3hSgrad63otqmyVKRKESNL2b+R3hVHfW+SLKNab+sZSmWqbREd6Iu0rxGOu1j1FTS9843Gy8YjouKPT4xUbEirU1/1uyKPbZlhWkqYPfQgONi01bn5Wp7ojgDKa0Ty5n/5vzy/T/5T5dX/Oby3dX5+5uQqabri7cX1xdXry/4m8vrkL2Fxos3/N313/nPl1dvPvx8E56wkb+b1+fvL6/e8bfnr28/XPOLqzf8zfntRdjvuLk9v75VXeN0bs/fXXzHby4u3gBL9PY98PTxA85Chm6vz69ugNblhyv++sPNLf/x401wcnISyzVbg1nwaitevPyLjyo6I80EbPJXVtXlGS0IZvA3UCUTbJfHMgU918kaDIat85IVZX4nM5GtJJkLjo+TjQQzmhkDiDT9gHrvk3pL1hDlhcx8r1x6AepgC9pI5ZndIhJfpvnqE0syloBZ+KnYLWNxpkdGpRSxf8p++IG9mAYhW3pecNYRkOIjaopY1NInWoqFUoIxZ6Z/Kx/Uk2+EQj7D0X0q32yWx0mphBOyvKmLpnZEVcTRG1GLt6XYSSuzf8gyWe9BwneSgTGQO1asqdDaS7nKyxjcsyjlBBwwybAV5SUrcsp1UoII07yqZGUFW+b3FYh1vjgxAkI5onwqsFAZd5iNNmm+9D3l8fwZ35QNbYk/i4raCxxZFWKf5iIGymT9Eb6QMYTgVQUHuZGjz7xV0Xghu5fJZltXPM/S/eytSCsZWFLIYSQKUGzsf9Zre2dmhblpWYTMQ17cLnpfjJs480BAdSJSrhbnKCl38lj3QVokWy6LfLXlKGGX0KDvIJXVVq4+FXmS1dp/gEzfm4IvSjRrNAyQr2snPspKdRMp1CyNizZl3hTLvdGdF0Ris/HJfGa+khPIL2uy5JdGekHLoGrhRhjKnGDKmHQepdCXAlEZiMYlEYA7VrKGtWP54LvbQpneSeVNvMBRWe0tYLOmm3q8RSR/8VMABBfOgi4htSFaAy2sT2h0+4pwb60uWbXDp8gOZQKkN7V/6qg4qnO+qu58BRDsOfOs18m7BCgDSkI/SI7k5HqPWu0RAi17K8kJoQ6RStYMgrPhf1T+oLoRYULrmCwWkUhTP1D/trABqFVJdg7jS4SHi7LMS98zSMdcfhWi6iTE64Cw4lIDL/og8hgnlI5U/lJUMk0y6aJvJpsS1GtbDiPwe0S1dd6UExW4HMqQ+txBprFB2FUE3V4mViUIQkG2hd8aE5PHAdjltw/ADv0I8iSwqS4MKx5nzHuPXvrux5881KSX4ttmufNwGYqbGYKJBHUz7zqJN9JrY2aLMxgduV5GYdEr1T33aCEycXrqzm4xGvoB3Hy7ZFQVaVL7HveC+ekCyCl5GLin2Uq1WqCPCapV4teK6Yk9FolckZKI76qWO4fvQ1sNh6KhrRMt3Gyo6c5fLCCfKCAZaWmprYyIAf2Q+lrmP8k9GZBZGTxOLQkPK3jAX8xW8LdOdvS7vo85OAv/brrzhsHI7ebgruWep2Jz6i3sSJlVcgeaUjKDpBbCuZZ/YGMNMgY+oTxfo0nI4jIvMqGxhaJQZ/1WVRAe2hfalRQZBBZmola3t0p+xZhhiYGkDJdtPNglmR+wP8zYICp007xxHBo4dSuHpGK7pKI8TK3VMtI1aTMjQjn4qzxtdlk10wwGHbBVE/vg+Bbw7iqv3+ZNFhu+cpcnk7BVCFNZDxpbbSnqAPWbLC+l1tFt2cjABc4Y4HXfgU9wL7mCpG5/FFJWycOkgiVQbCIDgKFMVOGkNE7dhUSDd6Hr8+Ejfm850m7/jMOqXK067u+POvSrx+HgKWiz/v3Koq/18/7cFho+t5h85qA12HVJcHxmcPlA6qj8dpcWOPKn9x9xJoR5fIPo6X2Z01ItA9pZ0dQUlyDOzzRfT/yiooFRRjCCRTZ2mUHfDKG+AdD8n0GL4ydaM/8daPl6GBAV8sXlQ13Knax8EvpZx+UJA+qmAHFAvj6HIn+BolRPCwsI12pt+Usj0skKsKlmy7yu8506MsoLFssVoBvbwTZkWW2TokUFLG1BKWAHxECEvs/vRNoAS/OevsimTEJO60BaIh7805CUo0kF7PlzdjrtyAWYNt1Qw0PJStODuSEZdkbUAI79EQY85YYjxRyQzl81ZQlh9Iy10gGG75K8qdo29m92lWeuLNeAwHjMhj+tFF+LdNWk4C1akKoAYzB1ci/2zCxKMm0FybAtFYWVJ9iw4QEtDJcGGaskX7PbBh4lnNNoGrJpNFX5GOgdnVoPBu4NOX1eU9V4gAMjqmbni2Xl+6dQe6D89ZQAeUCpIfYYMgQ6UxDCIZizVMxyHTJ2S5pOQPHC9BLPSt9aHGzmcsT+yFqqXV5dI5lGL9kzu8PQ0NKahyBQilhA0AQLXedpkle+Ez9HPGc0ev7YJGnMIO2aoGa+m04AXxpQeppnm0m1pbNGWSZ5rCIqxKkUUCq2+n/kgMdXMBwq5Mc8FCOzFp/Nkw24DlF74UZSYKHkCNqwBhrRK92U5veI2fS8BQvV3Z2jOJwGQtwVoXJqVY0AL5FKEv2qWYJ7zOYHE9I2BqjA0cPaFHepVx/FsaAzHCCxkqumxtO1mbs1cBB0DHISwirDOJt0JEBB6hYeQUy18JXKqtl306DPFoFDSE+uKTqgQby3siQjb9kj+ya86FDGvWrS9HiANvaFrnKOo67sv2ox2B7OKPQL5k9XDvi3oUJ4ZuhBkqWopOBSxFsQRBiRIZ5OxgeBROyYLovjh4XG4rUFnynTN+nGmWOJw2KIuOWKDRhKr3SqAaqDqhxA1woWuq1uh4SURoZTHKV5jkFoYvrtID072H0dGV3f5xyNYNnsOaRrqcvBC8Az33DO/txyZA4bh+7u2GbYdXPHq63/9XKP/oGlQk5Va6zSvJJa3F9RbSi8zMC9J0TBGqv6ILSVrMlE/K8Gkv0YbH0HOIPFEq2p8qgjjsQHlUZbXRx3mBAyk5bNbcZrMmBim1JgiB8owbqxx4ouXVU76AqiMxZPJU+DRSc3UssMqhYUVEfUVIVkRZTmG1+PUQwBuW2yrv3JKcbEbtf4Ib0aM9zgyJqLA1kpknskJ9VV6REBNlRjj42657owhXTcBnKmK8kQrShj2xzsjHYxqfNJa3DWfHQaPzsymvZroMUTlcwxVYxi+NE6xnKpHub6x9Nz8RxaDj9PgnHtZAk9lh0SL+RA2Ww+2Mc2v595qVyDewwi+ogxBF+dryhmvypVOSYvQf4HKYlNNJRD/a5E4zckAN8kzn91OB/R0rcItyTgJyPtt0kPRtZ5KnQfE6oPUB0N3f/DUD0Mz18Xk/HyBocKbifKva8Ljj7CrkuoRGW2ApSFQvYwuN4QGcAgk0Mh8YkNylqToS1h1FWSJQTRT/IOkBeHf01Z0zqw4vtInNDM2UJau8e8a54LY16m97DSFv1EtSQM+uxZweFnXvMMPD3lR32L8fQG0eygZlWAEx7IXAeTkTn+mMlZfYxPbQ8ZHAtXQpl3nKLvBwsjmEOe6lndW19cFpXZCWQjVkfP2OmUT6dTLGRsI1UxkNJkIvvSvQJSVGgWY3dYzvrwNl8DCNb8M8z5gotjmuSPGsThM4vu39P2AttBFp/rTQUj9mNRtzSh4SlPBtf7BBvR17R8Qsu+J1NeXCZVnh2bML0GDdWS8ms6h5rQ7QUGOemqTAoqJpURTzBJlc4K7dUeIEqH58DQWOoKOWGx17vHsXNvmLGqdp2VguA67zqBDVwSd3kq6iRN6j1/MY2JVJd2VOZpmmQb/8UUECXJuPax2QvQR1XHmqEiucvxeNHZFzUp+c5cpKklQmktytq+gQIPXgZpDyX0Oa5JF5V0Zl4psk88WXGV9Y3clehCZCk3EHwVA6F5AwaYPbSgos0/eFMME69HBpq7ZoEFZB5q3RMGo1QivHOFbLnf/DVHI6yo2XNXcos+F50xJM/WTyoYQJ8OSLVKvyo3jZayvpeAle3iQZvZaIAysjKvikM8SO4Kkd4CshIfxrjCDfo0NQGwKd8lO+muCemy2FcQ3anm8n01pN91clQW1rkg5UjyzAi+Fd0ZyqFrjnRXBM+xKu58oDKU8dMS4J59t9+tFiNUNiB5KIbpbhR+KeuzF9AZG5jKU5cdPTJySG02bWxTqnbcF0+dexFFhStwd956vzO1DwmjUcnTul0JiLSxKLnV21aUmNrq9y/HgTJeKrWQrH/7oDy4GYj1igXgj3gt1YVbvCyTZCusPQCIxUYkWVUTQEPE0VFAg/EdIJb6gmSweJ1sIOt6UN9G0xqkuEQWK/x08iLEbqwhZ/4pvH8fvQxaZxcPSWVON6CsEUtVof2aFD7SC9l8VPzhiDafQkT6m3s/ATkG5FTkYF2il9QGRJmtLlvEAS7zslLR434rS2lEP+8ZKrmMWJ4tAXE9/Vza20b4h9uOqpWo8TKpoaKksAhZh2zPi7B7pjgJOtkJwGapMFNPH0VNu7i+NS39tffzZ/QrTFr+pJzqT4vgC/BNTZYpfBllKOhtSzxs8auuP42m05egU3iu6n0qZ95k4oVKijOgJPb40VxJAkLQAynfp3/7goLuveqmS10TZhCDXUM0u3ytJQt2FtV0rzAVe7B/Hz/KQ1sl7iT89q6yWWvnOsch646KbANcxkUyO305xVs8YM+UEfhAwjhg2WS+KDdQ05ib5tEVfiUqxEoOne2dzGSJniPSFNAqm4Dey2TZ0L0Lc0W+lHgwpQ8e6ZIa+lh7LcM6G64bqa3QpaXdJ/jXByZgKxWdd0FgArHVPP+kj79oWucmMZDQ957ci7phn7jOVpzraLPhzTiaY6+bjdAbWUnfWDFnZbOnvn+BIswIvJhoDkdbO+nyjToeUjTJmD1+HbmxOLi5YnY8vNJCa/ab282pM+LZ8NRxQCkcPcN2yCNNkAC9HLv93rLH7F0X7J1LE/NOKW9kGjrixctgwSPY2yGgz/08Z8NesBg9s3U4MndRR7bpUufLPVcIdug+qthsIBKjK84s7ba8bwvrtsIfHqtSfM+aHchzpW6eO05m6B/L8bqBEk6p5uAdWopJww8DRFpFQbphhEVsrw0C8XqdPOAXRs8qXhW4hppl0qHULmsSX7U87Alf8TTO73SbOxOmmHG4tfTbXmcRHaoovR3WmW5lCeZv4poW7WPGr69wdokeEvBYNjWChDr1QrDn2OkT0A/xX8E+NeIpkB1wXm4aDDcfqccUgvQSiThGmtTvQ5w0aDoxmDkBFoD3el/Imbp7jLaaAIY41neAmILfb0Kq1ccElfhbyZD3T1oE/D0skTq/ATcDFkDboknr2cj/rHqUnLKZw/QG/y2qm/Mroq6ZgeWBa3OO91A4J2fmYNpJxrmnjA3TEXdGcPIf7hRubHw3AAA=","sh.000300_daily.parquet":"..."}
for name, payload in payloads.items():
    (project/name).write_bytes(gzip.decompress(base64.b64decode(payload)))
data_link = project/"data"
if data_link.is_symlink():
    data_link.unlink()
if not data_link.exists():
    data_link.symlink_to(source/"data", target_is_directory=True)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "captum>=0.7"])
from captum.attr import IntegratedGradients
assert torch.cuda.is_available(), "Enable GPU T4 x2"
print("STAGE4_CLEAN_ENV_OK", torch.__version__, torch.cuda.get_device_name(0), flush=True)


In [ ]:
v5 = Path("/kaggle/working/stage4_v5_download")
if v5.exists():
    shutil.rmtree(v5)
downloaded = Path(kagglehub.dataset_download("nudge147/a-share-5min-stage4-v5-immutable"))
shutil.copytree(downloaded, v5)
for archive_path in list(v5.rglob("*.zip")):
    try:
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(v5/"unzipped")
    except zipfile.BadZipFile:
        pass
artifacts = Path("/kaggle/working/stage4_v5_artifacts")
if artifacts.exists():
    shutil.rmtree(artifacts)
artifacts.mkdir()
for path in v5.rglob("*"):
    if not path.is_file() or "artifacts" not in path.parts:
        continue
    index = path.parts.index("artifacts")
    relative = Path(*path.parts[index + 1:])
    if relative.parts:
        target = artifacts/relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
assert len(list(artifacts.glob("*.pt"))) == 80
assert len(list(artifacts.glob("*.parquet"))) == 80
print("VERSION5_RESTORED", 80, "checkpoints", 80, "predictions", flush=True)


In [ ]:
attr_out = Path("/kaggle/working/stage4_attribution_clean")
attr_out.mkdir(exist_ok=True)
env = os.environ.copy()
# Kaggle currently pairs torch 2.10/cu128 with legacy P100 runners.  That
# wheel has no executable Pascal kernel, so attribution deliberately uses
# CPU for reproducibility across accelerator assignments.
env["CUDA_VISIBLE_DEVICES"] = ""
command = [sys.executable, "-u", str(project/"run_stage4_attribution.py"),
    "--windows", *map(str, range(1, 9)), "--artifact-dir", str(artifacts),
    "--flat-dir", str(project/"data/dataset/flat"),
    "--feature-dir", str(project/"data/features_5min"),
    "--output-dir", str(attr_out), "--device", "cpu",
    "--sample-count", "400", "--steps", "32", "--heartbeat-every", "50",
    "--window-timeout-seconds", "1200"]
subprocess.check_call(command, env=env)
assert len(list(attr_out.glob("window_*_gru_attribution_heatmap.csv"))) == 8
print("ATTRIBUTION_COMPLETE 8/8", flush=True)


In [ ]:
baseline = Path("/kaggle/working/stage4_baseline_clean")
if len(list((baseline/"artifacts").glob("*.joblib"))) != 16:
    subprocess.check_call([sys.executable, "-u", str(project/"run_baseline.py"),
        "--flat-dir", str(project/"data/dataset/flat"),
        "--feature-dir", str(project/"data/features_5min"),
        "--output-dir", str(baseline), "--save-predictions"])
prediction_dir = Path("/kaggle/working/stage4_six_signal_clean")
prediction_dir.mkdir(exist_ok=True)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = ""
print("INFERENCE_DEVICE cpu (portable fallback)", flush=True)
subprocess.check_call([sys.executable, "-u", str(project/"run_stage4_inference.py"),
    "--baseline-artifact-dir", str(baseline/"artifacts"),
    "--neural-artifact-dir", str(artifacts), "--output-dir", str(prediction_dir),
    "--flat-dir", str(project/"data/dataset/flat"),
    "--feature-dir", str(project/"data/features_5min"),
    "--seq-dir", str(project/"data/dataset/seq"), "--device", "cpu"], env=env)
assert len(list(prediction_dir.glob("*.parquet"))) == 96

daily = Path("/kaggle/working/daily_reference_from_5min")
daily.mkdir(exist_ok=True)
for path in (project/"data/features_5min").glob("*.parquet"):
    frame = pd.read_parquet(path, columns=["date", "code", "close", "time"])
    frame = frame.sort_values("time").groupby(["date", "code"], as_index=False).tail(1)
    frame["tradestatus"] = 1
    frame[["date", "code", "close", "tradestatus"]].to_parquet(daily/path.name, index=False)

comparison_parts = list(v5.rglob("four_model_comparison.csv"))
assert comparison_parts, "Version 5 comparison shards are missing"
comparison = Path("/kaggle/working/four_model_comparison_all_windows.csv")
pd.concat([pd.read_csv(p) for p in comparison_parts], ignore_index=True).drop_duplicates().to_csv(comparison, index=False)
index_path = project/"sh.000300_daily.parquet"
report = Path("/kaggle/working/stage4_final_report")
subprocess.check_call([sys.executable, "-u", str(project/"run_stage4_analysis.py"),
    "--baseline-artifact-dir", str(baseline/"artifacts"),
    "--neural-artifact-dir", str(artifacts), "--comparison-path", str(comparison),
    "--daily-prediction-dir", str(prediction_dir), "--index-path", str(index_path),
    "--daily-dir", str(daily), "--output-dir", str(report)])

durable = Path("/kaggle/working/stage4_clean_outputs")
durable.mkdir(exist_ok=True)
with zipfile.ZipFile(durable/"stage4_clean_complete.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for root in (attr_out, report, prediction_dir, baseline):
        for path in root.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to("/kaggle/working"))
kagglehub.dataset_upload("nudge147/a-share-5min-stage4-checkpoints", str(durable),
                         version_notes="clean notebook reproducibility run")
print("STAGE4_CLEAN_COMPLETE", flush=True)


In [ ]:
summary = pd.read_csv(report/"cost_summary_full_period.csv")
display(summary)
ranking = pd.concat([pd.read_csv(p) for p in attr_out.glob("window_*_gru_feature_ranking.csv")])
display(ranking.groupby("feature", as_index=False)["share"].mean().sort_values("share", ascending=False))
display(pd.read_csv(report/"gru_seed_independence_audit.csv"))
